# 09 — Kombinierte Query: MdB + Journalist:innen

Liest [`data/accounts.csv`](../data/accounts.csv) und baut **eine** Brandwatch-Query,
die die Inhalte aus Notebook 03 (MdB) und Notebook 06 (Journalist:innen) per `OR`
zusammenführt.

**Filter (Vereinigung beider Teile):**
- Teil A — MdB: `category == "MdB"`
- Teil B — Journalist:innen: `category == "News"` AND `label == "Journalist"`
- Beide: `channel ∈ {x, instagram, facebook}`

**Struktur:** MdB-Blöcke pro Partei alphabetisch (AfD, CDU, CSU, Grüne, Linke, SPD,
Sonstige Parteien), dann ein Journalist:innen-Block. Handles pro Block auf einer Zeile.

**Output:** `output/queries/mdb_journalists_query.txt`.

In [1]:
import os

import pandas as pd

if os.path.basename(os.getcwd()) == "scripts":
    PROJECT_ROOT = os.path.dirname(os.getcwd())
else:
    PROJECT_ROOT = os.getcwd()

DATA_DIR     = os.path.join(PROJECT_ROOT, "data")
QUERIES_DIR  = os.path.join(PROJECT_ROOT, "output", "queries")
ACCOUNTS_CSV = os.path.join(DATA_DIR, "accounts.csv")
OUTPUT_FILE  = os.path.join(QUERIES_DIR, "mdb_journalists_query.txt")

ALLOWED_CHANNELS = ["x", "instagram", "facebook"]
PARTY_ORDER = ["AfD", "CDU", "CSU", "Grüne", "Linke", "SPD", "Sonstige Parteien"]
LANGUAGE_FILTER = "language:de"

os.makedirs(QUERIES_DIR, exist_ok=True)

## 1. Daten laden + beide Teile filtern

In [2]:
accounts = pd.read_csv(ACCOUNTS_CSV)

def prep(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df[df["channel"].isin(ALLOWED_CHANNELS)]
        .dropna(subset=["handle"])
        .drop_duplicates(subset=["channel", "handle"])
        .copy()
    )

mdb = prep(accounts[accounts["category"] == "MdB"].dropna(subset=["label"]))
journos = prep(
    accounts[(accounts["category"] == "News") & (accounts["label"] == "Journalist")]
)

print(f"MdB:            {len(mdb):>5}")
print(f"Journalist:innen: {len(journos):>5}")
print(f"Gesamt:           {len(mdb) + len(journos):>5}")
print()
print("MdB-Parteien:")
print(mdb["label"].value_counts())

MdB:             1440
Journalist:innen:  1832
Gesamt:            3272

MdB-Parteien:
label
CDU                  385
SPD                  306
AfD                  298
Grüne                225
Linke                118
CSU                  104
Sonstige Parteien      4
Name: count, dtype: int64


## 2. Helper

In [3]:
def bw_author(handle: str) -> str:
    h = str(handle).strip().replace('"', '\\"')
    return f'author:"{h}"'


def block(comment: str, handles) -> str:
    body = " OR ".join(bw_author(h) for h in handles)
    return f"<<< {comment} — {len(handles)} Handles >>>\n({body})"


def sort_handles(series: pd.Series) -> list[str]:
    return series.sort_values(key=lambda s: s.str.lower()).tolist()


def party_order(labels):
    present = set(labels)
    ordered = [p for p in PARTY_ORDER if p in present]
    ordered += sorted(present - set(PARTY_ORDER))
    return ordered

## 3. Blöcke bauen

In [4]:
blocks: list[str] = []

# --- Teil A: MdB (pro Partei, Parteien in PARTY_ORDER-Reihenfolge) ---
mdb_party_count = 0
for party in party_order(mdb["label"].unique()):
    handles = sort_handles(mdb[mdb["label"] == party]["handle"])
    if not handles:
        continue
    blocks.append(block(f"MdB — {party}", handles))
    mdb_party_count += 1
print(f"MdB:              {mdb_party_count} Blöcke ({len(mdb)} Handles)")

# --- Teil B: Journalist:innen (ein Block) ---
if not journos.empty:
    blocks.append(block("Journalist:innen", sort_handles(journos["handle"])))
    print(f"Journalist:innen: 1 Block  ({len(journos)} Handles)")

print(f"\nBlöcke gesamt: {len(blocks)}")

MdB:              7 Blöcke (1440 Handles)
Journalist:innen: 1 Block  (1832 Handles)

Blöcke gesamt: 8


## 4. Gesamt-Query zusammensetzen

In [5]:
total_handles = sum(b.count('author:"') for b in blocks)
header = f"<<< MdB + Journalist:innen — {total_handles} Handles in {len(blocks)} Blöcken >>>"

body = "\nOR\n".join(blocks)
query = f"{header}\n({LANGUAGE_FILTER} AND (\n{body}\n))\n"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(query)

size = os.path.getsize(OUTPUT_FILE)
print(f"Datei:           {OUTPUT_FILE}")
print(f"Größe:           {size:,} bytes  ({size / 1024:.1f} KiB)")
print(f"Handles gesamt:  {total_handles}")
print(f"Blöcke:          {len(blocks)}")

if size > 100_000:
    print(f"\n⚠️  {size - 100_000:,} Zeichen über dem 100k-Limit.")

Datei:           /Users/zorbeyozcan/Projekte/query_printer/output/queries/mdb_journalists_query.txt
Größe:           83,637 bytes  (81.7 KiB)
Handles gesamt:  3272
Blöcke:          8


## 5. Preview

In [6]:
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if "<<<" in line and ">>>" in line:
            print(line.rstrip())

<<< MdB + Journalist:innen — 3272 Handles in 8 Blöcken >>>
<<< MdB — AfD — 298 Handles >>>
<<< MdB — CDU — 385 Handles >>>
<<< MdB — CSU — 104 Handles >>>
<<< MdB — Grüne — 225 Handles >>>
<<< MdB — Linke — 118 Handles >>>
<<< MdB — SPD — 306 Handles >>>
<<< MdB — Sonstige Parteien — 4 Handles >>>
<<< Journalist:innen — 1832 Handles >>>
